# One File

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date, input_file_name, regexp_extract
import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("SIM Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [38]:
import os
from pyspark.sql import SparkSession

BASE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/conversion/"
WRITE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/conversion/one_file/"

lista_base = ['ano_2018', 'ano_2020', 'ano_2022']
#lista_base = ['ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022']

dfs = []
for ano in lista_base:

    print(f"Ano de {ano}")
    
    # 1. ler pasta do ano
    df = spark.read.parquet(f"{BASE_PATH}{ano}/")
    
    # 2. validação de leitura
    pessoal.completude(df)
    
    # 3. consolidar em 1 arquivo
    df.coalesce(1).write.mode("overwrite").parquet(f"{WRITE_PATH}/{ano}")
    
    # 4. ler de volta e validar
    df_unico = spark.read.parquet(f"{WRITE_PATH}/{ano}")
    pessoal.completude(df_unico)

Ano de ano_2018


Qtd. registros: 1316719 | Quantidade de colunas:  88


Qtd. registros: 1316719 | Quantidade de colunas:  88
Ano de ano_2020
Qtd. registros: 1556824 | Quantidade de colunas:  88


Qtd. registros: 1556824 | Quantidade de colunas:  88
Ano de ano_2022
Qtd. registros: 1544266 | Quantidade de colunas:  88


Qtd. registros: 1544266 | Quantidade de colunas:  88


## Confirmando o tipo da mesma variavel nos parquets do ano desejado

In [15]:
import os
from pyspark.sql import SparkSession

BASE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/conversion/"

In [16]:
import os

path = f"{BASE_PATH}ano_2021/"
files = os.listdir(path)

for f in files:
    df = spark.read.parquet(path + f)
    print(f)
    df.select("CAUSAMAT").printSchema()

part_4.parquet
root
 |-- CAUSAMAT: integer (nullable = true)

part_8.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_5.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_2.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_1.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_9.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_3.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_0.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_7.parquet
root
 |-- CAUSAMAT: string (nullable = true)

part_6.parquet
root
 |-- CAUSAMAT: string (nullable = true)



## Corrigindo tipo de variável de um dos arquivos parquet

In [17]:
BASE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/conversion/ano_2021/part_4.parquet"

In [18]:
df = spark.read.parquet(BASE_PATH)
df.printSchema()

root
 |-- index: string (nullable = true)
 |-- ORIGEM: string (nullable = true)
 |-- TIPOBITO: string (nullable = true)
 |-- DTOBITO: string (nullable = true)
 |-- HORAOBITO: string (nullable = true)
 |-- NATURAL: string (nullable = true)
 |-- CODMUNNATU: string (nullable = true)
 |-- DTNASC: string (nullable = true)
 |-- IDADE: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- RACACOR: string (nullable = true)
 |-- ESTCIV: string (nullable = true)
 |-- ESC: string (nullable = true)
 |-- ESC2010: string (nullable = true)
 |-- SERIESCFAL: string (nullable = true)
 |-- OCUP: string (nullable = true)
 |-- CODMUNRES: string (nullable = true)
 |-- LOCOCOR: string (nullable = true)
 |-- CODESTAB: string (nullable = true)
 |-- ESTABDESCR: integer (nullable = true)
 |-- CODMUNOCOR: string (nullable = true)
 |-- IDADEMAE: string (nullable = true)
 |-- ESCMAE: string (nullable = true)
 |-- ESCMAE2010: string (nullable = true)
 |-- SERIESCMAE: string (nullable = true)
 |-- OCUPMAE

In [19]:
from pyspark.sql import functions as F

df = df.withColumn("CAUSAMAT", F.col("CAUSAMAT").cast("string"))
df.printSchema()

root
 |-- index: string (nullable = true)
 |-- ORIGEM: string (nullable = true)
 |-- TIPOBITO: string (nullable = true)
 |-- DTOBITO: string (nullable = true)
 |-- HORAOBITO: string (nullable = true)
 |-- NATURAL: string (nullable = true)
 |-- CODMUNNATU: string (nullable = true)
 |-- DTNASC: string (nullable = true)
 |-- IDADE: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- RACACOR: string (nullable = true)
 |-- ESTCIV: string (nullable = true)
 |-- ESC: string (nullable = true)
 |-- ESC2010: string (nullable = true)
 |-- SERIESCFAL: string (nullable = true)
 |-- OCUP: string (nullable = true)
 |-- CODMUNRES: string (nullable = true)
 |-- LOCOCOR: string (nullable = true)
 |-- CODESTAB: string (nullable = true)
 |-- ESTABDESCR: integer (nullable = true)
 |-- CODMUNOCOR: string (nullable = true)
 |-- IDADEMAE: string (nullable = true)
 |-- ESCMAE: string (nullable = true)
 |-- ESCMAE2010: string (nullable = true)
 |-- SERIESCMAE: string (nullable = true)
 |-- OCUPMAE

In [20]:
df.write.mode("overwrite").parquet("/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/analytics/")

## Criando o parquet unico por ano

In [22]:
from pyspark.sql import functions as F

BASE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/conversion/"
WRITE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/conversion/one_file2/"

ano = "ano_2021"  # trocar aqui para testar ano por ano

print(f"\n Processando: {ano}")

path = f"{BASE_PATH}{ano}/"

# 1. leitura FORÇANDO leitura permissiva
df = spark.read.parquet(path)

# 2. debug: checar schema real
df.printSchema()

# 3. normalização FORTE (resolve INT32 vs STRING)
df = df.withColumn(
    "CAUSAMAT",
    F.col("CAUSAMAT").cast("string")
)

# 4. validação
pessoal.completude(df)

# 5. escrita
output_path = f"{WRITE_PATH}/{ano}"
df.coalesce(1).write.mode("overwrite").parquet(output_path)

# 6. leitura final (teste)
df_check = spark.read.parquet(output_path)
pessoal.completude(df_check)

print(" Ano processado com sucesso")


 Processando: ano_2021


root
 |-- index: string (nullable = true)
 |-- ORIGEM: string (nullable = true)
 |-- TIPOBITO: string (nullable = true)
 |-- DTOBITO: string (nullable = true)
 |-- HORAOBITO: string (nullable = true)
 |-- NATURAL: string (nullable = true)
 |-- CODMUNNATU: string (nullable = true)
 |-- DTNASC: string (nullable = true)
 |-- IDADE: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- RACACOR: string (nullable = true)
 |-- ESTCIV: string (nullable = true)
 |-- ESC: string (nullable = true)
 |-- ESC2010: string (nullable = true)
 |-- SERIESCFAL: string (nullable = true)
 |-- OCUP: string (nullable = true)
 |-- CODMUNRES: string (nullable = true)
 |-- LOCOCOR: string (nullable = true)
 |-- CODESTAB: string (nullable = true)
 |-- ESTABDESCR: integer (nullable = true)
 |-- CODMUNOCOR: string (nullable = true)
 |-- IDADEMAE: string (nullable = true)
 |-- ESCMAE: string (nullable = true)
 |-- ESCMAE2010: string (nullable = true)
 |-- SERIESCMAE: string (nullable = true)
 |-- OCUPMAE

Qtd. registros: 1832649 | Quantidade de colunas:  88
 Ano processado com sucesso
